# Extração dos textos 

O objetivo deste código é extrair o texto do PDF de todos os processos cujos PCTCs estavam disponíveis. Como resultado, teremos uma base de dados com duas colunas: ID do processo e texto, que sera uma entrada string que contém todo o texto dos processos.

## Importando e baixando os pacotes

Importamos os pacotes que vamos utilizar:

In [11]:
!pip install pdfplumber

In [12]:
import os
import pandas as pd
import re
import random
import shutil
import pdfplumber
import numpy as np
import time
from pdf2image import convert_from_path
from PIL import Image
import pytesseract
from fpdf import FPDF

Definimos o caminho da pasta com os arquivos:

In [13]:
pasta_principal = "D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/raw/pdfs/pareceres_do_ctc"

## Extração dos textos e criação do df

In [54]:
def extrair_texto_pdf(caminho_pdf, min_chars=10000):

    # 1️⃣ Número do processo a partir do nome do arquivo
    numero_processo = os.path.basename(caminho_pdf)
    
    # Inicializar a variável escaneado
    escaneado = False
    
    # 2️⃣ Tenta extrair texto normalmente
    with pdfplumber.open(caminho_pdf) as pdf:
        texto = "\n".join(page.extract_text() or "" for page in pdf.pages)
    
    # 3️⃣ Se texto muito curto ou vazio, tentar OCR
    if len(texto.strip()) < min_chars:
        print(f"⚠️ Texto curto para {numero_processo}, tentando OCR...")
        try:
            imagens = convert_from_path(caminho_pdf)
            texto_ocr = "\n".join(pytesseract.image_to_string(img, lang='por') for img in imagens)
            
            # Se OCR encontrou mais texto, substitui
            if len(texto_ocr.strip()) > len(texto.strip()) + 100:
                texto = texto_ocr
                escaneado = True
            # Se não encontrou mais texto, mantém escaneado = False
                
        except Exception as e:
            print(f"❌ Falha no OCR para {numero_processo}: {e}")
            # escaneado permanece False em caso de erro
    
    return numero_processo, texto, escaneado

In [55]:
# Lista para acumular os dados
dados = []
# Iniciar o timer total
start_time = time.time()

# Loop pelas subpastas de anos (2000 a 2023)
for arquivo in os.listdir(pasta_principal):
    if arquivo.endswith('.pdf'):
        caminho_pdf = os.path.join(pasta_principal, arquivo)
        # Timer para cada iteração
        iter_start = time.time()
        try:
            # Extrair informações do PDF (agora retorna 3 valores)
            numero_processo, texto, escaneado = extrair_texto_pdf(caminho_pdf)
            
            # Adicionar dados à lista
            dados.append({
                "ID": numero_processo,
                "Texto": texto,
                "Escaneado": escaneado
            })
        except Exception as e:
            print(f"Erro ao processar {arquivo} na pasta {pasta_principal}: {e}")
            continue
        
        # Timer de cada iteração
        iter_end = time.time()
        iter_time = iter_end - iter_start
        print(f"Tempo para processar {arquivo} da pasta {pasta_principal}: {iter_time:.2f} segundos")

# Criar o DataFrame com todos os dados
df = pd.DataFrame(dados)

# Calcular o tempo total
end_time = time.time()
execution_time = end_time - start_time
print(f"Tempo total de execução: {execution_time:.2f} segundos")

# Exibir o DataFrame com todos os resultados
print(f"Total de PDFs processados: {len(df)}")
print(f"PDFs que precisaram de OCR: {df['Escaneado'].sum()}")
df

⚠️ Texto curto para 01_2009-RJ2012_12067.pdf, tentando OCR...
Tempo para processar 01_2009-RJ2012_12067.pdf da pasta D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/raw/pdfs/pareceres_do_ctc: 9.14 segundos
Tempo para processar 01_2012 - RJ2013_8159.pdf da pasta D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/raw/pdfs/pareceres_do_ctc: 1.95 segundos
Tempo para processar 02_01 - CIA. PAULISTA DE FERRO E GÁS.pdf da pasta D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/raw/pdfs/pareceres_do_ctc: 1.36 segundos
Tempo para processar 02_01 - CIA. PAULISTA DE FERRO E GÁS_1.pdf da pasta D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/raw/pdfs/pareceres_do_ctc: 1.63 segundos
Tempo para processar 04_01 E - SP 2002_0235 - BM&F.pdf da pasta D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/raw/pdfs/pareceres_do_ctc: 2.44 segundos

,ID,Texto,Escaneado
0,01_2009-RJ2012_12067.pdf,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False
1,01_2012 - RJ2013_8159.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False
2,02_01 - CIA. PAULISTA DE FERRO E GÁS.pdf,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False
3,02_01 - CIA. PAULISTA DE FERRO E GÁS_1.pdf,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False
4,04_01 E - SP 2002_0235 - BM&F.pdf,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False
...,...,...,...
740,SP2013_12 - RJ2013_8604.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False
741,SP2013_157.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False
742,SP2013_260.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False
743,SP2013_295.pdf,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False


## Ajustes na coluna "ID"

Começamos removendo duplicatas:

In [57]:
df = df.drop_duplicates()

df

,ID,Texto,Escaneado
0,01_2009-RJ2012_12067.pdf,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False
1,01_2012 - RJ2013_8159.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False
2,02_01 - CIA. PAULISTA DE FERRO E GÁS.pdf,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False
3,02_01 - CIA. PAULISTA DE FERRO E GÁS_1.pdf,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False
4,04_01 E - SP 2002_0235 - BM&F.pdf,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False
...,...,...,...
740,SP2013_12 - RJ2013_8604.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False
741,SP2013_157.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False
742,SP2013_260.pdf,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False
743,SP2013_295.pdf,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False


Depois, sabendo que temos entradas duplicadas/pdfs que foram baixados mais de uma vez, vamos remover o _X que aparece ao final desses arquivos:

In [58]:
pd.set_option("display.max_colwidth", 100)

df["ID"] = df["ID"].str.replace(r'(_[1-6])?\.pdf$|_[1-6]$', '', regex=True)

df

,ID,Texto,Escaneado
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False
...,...,...,...
740,SP2013_12 - RJ2013_8604,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False
741,SP2013_157,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False
742,SP2013_260,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False
743,SP2013_295,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False


Criamos uma coluna nova, aplicando uma função que extrai o número do processo para as entraas da coluna "ID" que contiverem alguma dos padrões:

In [59]:
# Função para verificar e transformar os processos
def formatar_identificador(identificador):
    # Expressões regulares para os diferentes formatos
    padroes = {
        r"^\d{5}\.\d{6}_\d{4}-\d{2}$": lambda x: f"{x[:5]}-{x[6:12]}-{x[13:]}",  # padrão
        r"^\d{2}_\d{2}$": lambda x: f"IA-20{x[3:]}-{x[:2]}",
        r"^\d{2}_\d{4}$": lambda x: f"IA-{x[3:]}-{x[:2]}",
        r"^RJ\d{4}_\d{2}$": lambda x: f"RJ-{x[2:6]}-{x[7:].zfill(5)}",
        r"^RJ\d{4}_\d{3}$": lambda x: f"RJ-{x[2:6]}-{x[7:].zfill(5)}",
        r"^RJ\d{4}_\d{4}$": lambda x: f"RJ-{x[2:6]}-{x[7:].zfill(5)}",
        r"^RJ\d{4}_\d{5}$": lambda x: f"RJ-{x[2:6]}-{x[7:]}",
        r"^SP\d{4}_\d{2}$": lambda x: f"SP-{x[2:6]}-{x[7:].zfill(5)}",
        r"^SP\d{4}_\d{3}$": lambda x: f"SP-{x[2:6]}-{x[7:].zfill(5)}",
        r"^SP\d{4}_\d{4}$": lambda x: f"SP-{x[2:6]}-{x[7:].zfill(5)}",
        r"^SP\d{4}_\d{5}$": lambda x: f"SP-{x[2:6]}-{x[7:]}"
    }
    
    # Verificar se o identificador segue algum dos padrões e formatar
    for regex, transform in padroes.items():
        if re.fullmatch(regex, identificador.strip()):  # Garantindo que checamos a string inteira
            return transform(identificador)
    
    # Retornar None se não for válido
    return None

# Aplicar a função à coluna do dataframe
df['identificador_formatado'] = df['ID'].apply(formatar_identificador)

# Exibir os resultados
df

,ID,Texto,Escaneado,identificador_formatado
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False,None
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False,None
...,...,...,...,...
740,SP2013_12 - RJ2013_8604,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False,None
741,SP2013_157,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False,SP-2013-00157
742,SP2013_260,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False,SP-2013-00260
743,SP2013_295,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False,SP-2013-00295


Para quais linhas isso não funcionou?

In [60]:
df[df['identificador_formatado'].isna()]

,ID,Texto,Escaneado,identificador_formatado
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False,None
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False,None
...,...,...,...,...
730,SP2005_268 - BM&F,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nREF.: PROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP20...,False,None
731,SP2005_268 - BOAVISTA,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nREF.: PROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP20...,False,None
732,SP2005_268 - SANOS,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nREF.: PROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP20...,False,None
738,SP2011_284 - RJ2013_2758,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP 2011/28...,False,None


Notamos que não funcionou para os casos em que a mesma entrada se refere a mais de um processo.

Para esses casos, vamos usar separadores para, quando tivermos dois processos, eles serem adicionados a uma lista na coluna "identificador_lista".

In [61]:
def separar_identificadores(identificador):
    """
    Separa uma string com dois ou mais processos em uma lista de identificadores únicos.
    """
    # Expressões regulares para encontrar os separadores comuns
    separadores = r"(?i)\s+-\s+|\s+e\s+|\s+E\s+"  # " - ", " e ", " E " com tratamento case insensitive

    # Separar os identificadores com base nos separadores
    partes = re.split(separadores, identificador)

    # Limpar espaços em branco extras e retornar apenas os elementos não vazios
    return [parte.strip() for parte in partes if parte.strip()]

# Aplicar a função 'separar_identificadores' na coluna 'Número do processo'
df['identificador_lista'] = df['ID'].apply(separar_identificadores)

# Exibir o DataFrame com a nova coluna
df

,ID,Texto,Escaneado,identificador_formatado,identificador_lista
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None,[01_2009-RJ2012_12067]
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False,None,"[01_2012, RJ2013_8159]"
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,"[02_01, CIA. PAULISTA DE FERRO, GÁS]"
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,"[02_01, CIA. PAULISTA DE FERRO, GÁS]"
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False,None,"[04_01, - SP 2002_0235, BM&F]"
...,...,...,...,...,...
740,SP2013_12 - RJ2013_8604,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False,None,"[SP2013_12, RJ2013_8604]"
741,SP2013_157,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False,SP-2013-00157,[SP2013_157]
742,SP2013_260,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False,SP-2013-00260,[SP2013_260]
743,SP2013_295,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False,SP-2013-00295,[SP2013_295]


Vamos ver se em alguma linha, não obtivemos nada para a entrada de "identificador_lista".

In [62]:
df[df['identificador_lista'].isna()]

,ID,Texto,Escaneado,identificador_formatado,identificador_lista


In [63]:
# Função para verificar se o item contém números
def contains_number(item):
    return any(char.isdigit() for char in item)

# Aplica a função para filtrar as listas na coluna 'listas'
df['identificador_lista'] = df['identificador_lista'].apply(lambda lista: [item for item in lista if contains_number(item)])

df

,ID,Texto,Escaneado,identificador_formatado,identificador_lista
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None,[01_2009-RJ2012_12067]
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False,None,"[01_2012, RJ2013_8159]"
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,[02_01]
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,[02_01]
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False,None,"[04_01, - SP 2002_0235]"
...,...,...,...,...,...
740,SP2013_12 - RJ2013_8604,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False,None,"[SP2013_12, RJ2013_8604]"
741,SP2013_157,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False,SP-2013-00157,[SP2013_157]
742,SP2013_260,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False,SP-2013-00260,[SP2013_260]
743,SP2013_295,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False,SP-2013-00295,[SP2013_295]


Agora, vamos pegar a coluna "identificador_lista" e iterar sobre ela, tendo como output a coluna "identificador_lista_transformada", que tem várias listas também, mas cujas entradas são no formato padronizado dos processos.

In [64]:
# Definição dos padrões e transformações
padroes = {
     r"^\d{5}\.\d{6}_\d{4}-\d{2}$": lambda x: f"{x[:5]}-{x[6:12]}-{x[13:]}",  # padrão
        r"^\d{2}_\d{2}$": lambda x: f"IA-20{x[3:]}-{x[:2]}",
        r"^\d{2}_\d{4}$": lambda x: f"IA-{x[3:]}-{x[:2]}",
        r"^RJ\d{4}_\d{2}$": lambda x: f"RJ-{x[2:6]}-{x[7:].zfill(5)}",
        r"^RJ\d{4}_\d{3}$": lambda x: f"RJ-{x[2:6]}-{x[7:].zfill(5)}",
        r"^RJ\d{4}_\d{4}$": lambda x: f"RJ-{x[2:6]}-{x[7:].zfill(5)}",
        r"^RJ\d{4}_\d{5}$": lambda x: f"RJ-{x[2:6]}-{x[7:]}",
        r"^SP\d{4}_\d{2}$": lambda x: f"SP-{x[2:6]}-{x[7:].zfill(5)}",
        r"^SP\d{4}_\d{3}$": lambda x: f"SP-{x[2:6]}-{x[7:].zfill(5)}",
        r"^SP\d{4}_\d{4}$": lambda x: f"SP-{x[2:6]}-{x[7:].zfill(5)}",
        r"^SP\d{4}_\d{5}$": lambda x: f"SP-{x[2:6]}-{x[7:]}"
}

def transformar_coluna(df, coluna):
    # Nova coluna processada
    novas_listas = []

    for idx, entrada in enumerate(df[coluna]):
        nova_lista = []

        for item in entrada:
            # Extrair todos os possíveis matches de padrões em cada string
            for padrao, transformacao in padroes.items():
                matches = re.findall(padrao, item)
                for match in matches:
                    nova_lista.append(transformacao(match))

        if not nova_lista:
            # Se nenhum padrão foi encontrado em toda a string
            print(f"Nenhum padrão encontrado na entrada: '{entrada}' no índice {idx}")

        novas_listas.append(nova_lista)

    # Atualiza a coluna no dataframe
    df[f"{coluna}_transformada"] = novas_listas

# Chama a função para transformar a coluna
transformar_coluna(df, "identificador_lista")

# Exibe o dataframe atualizado
df

Nenhum padrão encontrado na entrada: '['01_2009-RJ2012_12067']' no índice 0
Nenhum padrão encontrado na entrada: '['04_2014 (19957.000633_2015-31)']' no índice 6
Nenhum padrão encontrado na entrada: '['07_2014 (19957.000511_2015-45)']' no índice 16
Nenhum padrão encontrado na entrada: '['19957. 000441_2022-54', '19957. 009424_2021-00']' no índice 52
Nenhum padrão encontrado na entrada: '['19957.004471_2019-34 (ADITAMENTO TC 28)']' no índice 176
Nenhum padrão encontrado na entrada: '['19957.005332_2018-47\u200b']' no índice 207
Nenhum padrão encontrado na entrada: '['19957.005332_2018-47\u200b']' no índice 208
Nenhum padrão encontrado na entrada: '['19957.005332_2018-47\u200b']' no índice 209
Nenhum padrão encontrado na entrada: '['19957.006799-2019-95']' no índice 266
Nenhum padrão encontrado na entrada: '['19957.006910_2019-3']' no índice 270
Nenhum padrão encontrado na entrada: '['19957.0080352020-78']' no índice 295
Nenhum padrão encontrado na entrada: '['19957.009192_2018-86\u200b'

,ID,Texto,Escaneado,identificador_formatado,identificador_lista,identificador_lista_transformada
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None,[01_2009-RJ2012_12067],[]
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False,None,"[01_2012, RJ2013_8159]","[IA-2012-01, RJ-2013-08159]"
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,[02_01],[IA-2001-02]
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,[02_01],[IA-2001-02]
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False,None,"[04_01, - SP 2002_0235]",[IA-2001-04]
...,...,...,...,...,...,...
740,SP2013_12 - RJ2013_8604,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False,None,"[SP2013_12, RJ2013_8604]","[SP-2013-00012, RJ-2013-08604]"
741,SP2013_157,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False,SP-2013-00157,[SP2013_157],[SP-2013-00157]
742,SP2013_260,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False,SP-2013-00260,[SP2013_260],[SP-2013-00260]
743,SP2013_295,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False,SP-2013-00295,[SP2013_295],[SP-2013-00295]


Para casos omissos, fazemos manualmente.

In [65]:
# Verifica se a coluna "identificador_lista_transformada" existe; caso contrário, cria com listas vazias
if "identificador_lista_transformada" not in df.columns:
    df["identificador_lista_transformada"] = [[]] * len(df)

# Agora insere as listas manualmente na coluna, conforme os casos levantados
df.at[0,   "identificador_lista_transformada"] = ["RJ-2009-01", "RJ-2012-12067"]
df.at[6,   "identificador_lista_transformada"] = ["IA-2014-04", "19957-000633-2015-31"]
df.at[16,  "identificador_lista_transformada"] = ["IA-2014-07", "19957-000511-2015-45"]
df.at[52,  "identificador_lista_transformada"] = ["19957-000441-2022-54", "19957-009424-2021-00"]
df.at[176, "identificador_lista_transformada"] = ["19957-004471-2019-34"]
df.at[207, "identificador_lista_transformada"] = ["19957-005332-2018-47"]
df.at[208, "identificador_lista_transformada"] = ["19957-005332-2018-47"]
df.at[209, "identificador_lista_transformada"] = ["19957-005332-2018-47"]
df.at[266, "identificador_lista_transformada"] = ["19957-006799-2019-95"]
df.at[270, "identificador_lista_transformada"] = ["19957-006910-2019-43"]
df.at[295, "identificador_lista_transformada"] = ["19957-008035-2020-78"]
df.at[326, "identificador_lista_transformada"] = ["19957-009192-2018-86"]
df.at[370, "identificador_lista_transformada"] = ["19957-010559-2018-12"]
df.at[378, "identificador_lista_transformada"] = ["19957-011091-2019-56"]
df.at[628, "identificador_lista_transformada"] = ["RJ-2012-11199", "RJ-2013-06059"]
df.at[629, "identificador_lista_transformada"] = ["RJ-2012-11199", "RJ-2013-06059"]
df.at[630, "identificador_lista_transformada"] = ["RJ-2012-11199", "RJ-2013-06059"]
df.at[631, "identificador_lista_transformada"] = ["RJ-2012-11199", "RJ-2013-06059"]

# Exibe as alterações para conferir
df.loc[[0, 6, 16, 51, 173, 204, 205, 206, 254, 258, 280, 311, 350, 357, 703, 705, 709, 728]]

,ID,Texto,Escaneado,identificador_formatado,identificador_lista,identificador_lista_transformada
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None,[01_2009-RJ2012_12067],"[RJ-2009-01, RJ-2012-12067]"
6,04_2014 (19957.000633_2015-31),"COMISSÃO DE VALORES MOBILIÁRIOS\nRua Sete de Setembro, 111/2-5º e 23-34º Andares, Centro, Rio de...",False,None,[04_2014 (19957.000633_2015-31)],"[IA-2014-04, 19957-000633-2015-31]"
16,07_2014 (19957.000511_2015-45),Comissão de 'vá/ores Mobiliários\nProtegendo quem investe no futuro do Brasil\nPARECERDO COMITÊ ...,False,None,[07_2014 (19957.000511_2015-45)],"[IA-2014-07, 19957-000511-2015-45]"
51,18_05,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nREF.: PROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 18/2...,False,IA-2005-18,[18_05],[IA-2005-18]
173,19957.004386_2022-71,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRAT...,False,19957-004386-2022-71,[19957.004386_2022-71],[19957-004386-2022-71]
204,19957.005290_2019-25,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO COMITÊ DE TERMO DE COMPROMISSO\nSUMÁRIO\nPROCESSO AD...,False,19957-005290-2019-25,[19957.005290_2019-25],[19957-005290-2019-25]
205,19957.005313_2018-11,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO COMITÊ DE TERMO DE COMPROMISSO\nSUMÁRIO\nPROCESSO AD...,False,19957-005313-2018-11,[19957.005313_2018-11],[19957-005313-2018-11]
206,19957.005332_2018-47,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRAT...,False,19957-005332-2018-47,[19957.005332_2018-47],[19957-005332-2018-47]
254,19957.006551_2020-68,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRAT...,False,19957-006551-2020-68,[19957.006551_2020-68],[19957-006551-2020-68]
258,19957.006602_2018-37 e 19957.008430_2018-36,COMISSÃO DE VALORES MOBILIÁRIOS\nPARECER DO COMITÊ DE TERMO DE COMPROMISSO\nSUMÁRIO\nPARECER CON...,False,None,"[19957.006602_2018-37, 19957.008430_2018-36]","[19957-006602-2018-37, 19957-008430-2018-36]"


Checamos que não temos nenhuma lista vazia.

In [66]:
vazias = df["identificador_lista_transformada"].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
print("Número de listas vazias:", vazias)

Número de listas vazias: 0


Selecionamos apenas as colunas que vamos usar.

In [67]:
df

,ID,Texto,Escaneado,identificador_formatado,identificador_lista,identificador_lista_transformada
0,01_2009-RJ2012_12067,PROCESSO DE TERMO DE COMPROMISSO CVM Nº RJ2012/12067\nReg. Col. 8327/2012\nInteressado: Sérgio R...,False,None,[01_2009-RJ2012_12067],"[RJ-2009-01, RJ-2012-12067]"
1,01_2012 - RJ2013_8159,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº 01/12\nPRO...,False,None,"[01_2012, RJ2013_8159]","[IA-2012-01, RJ-2013-08159]"
2,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,[02_01],[IA-2001-02]
3,02_01 - CIA. PAULISTA DE FERRO E GÁS,INQUÉRITO ADMINISTRATIVO CVM Nº 02/01\nASSUNTO: APRECIAÇÃO DE PROPOSTA DE TERMO DE COMPROMISSO\n...,False,None,[02_01],[IA-2001-02]
4,04_01 E - SP 2002_0235 - BM&F,Processos Administrativos Sancionadores CVM 04/01 e 2002/0235\nAssunto: Apreciação de propostas ...,False,None,"[04_01, - SP 2002_0235]",[IA-2001-04]
...,...,...,...,...,...,...
740,SP2013_12 - RJ2013_8604,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM SP2013/12\nPR...,False,None,"[SP2013_12, RJ2013_8604]","[SP-2013-00012, RJ-2013-08604]"
741,SP2013_157,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO SANCIONADOR CVM Nº SP2013/015...,False,SP-2013-00157,[SP2013_157],[SP-2013-00157]
742,SP2013_260,PARECER DO COMITÊ DE TERMO DE COMPROMISSO\nPROCESSO ADMINISTRATIVO CVM Nº SP 2013/260\nRELATÓRIO...,False,SP-2013-00260,[SP2013_260],[SP-2013-00260]
743,SP2013_295,PARRECER DOO COMITÊÊ DE TERMMO DE COOMPROMIISSO\nPRROCESSOO ADMINISSTRATIVOO CVM Nºº SP 2013/295...,False,SP-2013-00295,[SP2013_295],[SP-2013-00295]


In [68]:
df = df[["ID", "Texto", "identificador_lista_transformada", 'Escaneado']]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 745 entries, 0 to 744
Data columns (total 4 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   ID                                745 non-null    object
 1   Texto                             745 non-null    object
 2   identificador_lista_transformada  745 non-null    object
 3   Escaneado                         745 non-null    bool  
dtypes: bool(1), object(3)
memory usage: 18.3+ KB


Exportamos como CSV:

In [69]:
df.to_csv("D:/Backup Tomas/T0mas/Faculdade/IC/Geral/Github CVM/corporate-fraud-in-brazil/data/interim/textos_dos_processos/textos_pctcs.csv", index=False)